## 5. Código Unificado para notebooks/05_Control_PID_dinamico.ipynb

El siguiente código unifica de forma secuencial y ejecutable el contenido íntegro de **05_control_PID_Lazo_abiertocerrado.ipynb**, **05_control_pid_perturbaciones.ipynb** y el análisis de estabilidad de Routh-Hopf, Frente de Pareto e integración NPT-3:

In [ ]:
# %% [markdown]
# # 05 · Control PID Dinámico Integrado (A3110 + Tren NPT-3 Áreas 2000–3000–6000)
# Integra: (1) Lazo abierto (+/-, con y sin ruido), (2) Lazo cerrado y consistencia estacionaria,
# (3) Cambios de consigna (IMC ε, ponderación β y ruido), (4) Estabilidad Routh-Hurwitz/Hopf y Frente de Pareto,
# y (Parte B) Subsistema de dilución AIT 3105 -> FFIC 3102 -> FIRC 3103 + Purga AIC 3255.

# %%
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from msmpr_core.params import make_params
from msmpr_core.plant import Plant, simulate, simulate_plant_npt3_calibrated
from msmpr_core import control as C
from msmpr_core.control import (
    ConsignaVariable, Ruido, imc_pi, iae, metricas_consigna, sigma_ventana, routh_hopf_analysis
)

os.makedirs("../docs/fig", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

# Estado estacionario nominal A3110 y sintonía base
p = make_params()
pl = Plant(p)
x0, u0, d0 = pl.steady_state(alpha=0.25)
m0 = pl.metrics(x0, u0, d0)
q0, qs0 = u0["Q_dil_cmd"], d0["Q_stock"]
R0 = q0 / qs0
DT, TP, H_OL = 5.0, 3600.0, 18 * 3600
tun, info = C.autotune(pl, x0, u0, d0, dt_s=DT)
sp = {"T": x0[1], "V": x0[0], "rho": 0.0}
RUIDO = Ruido(seed=7, dt=DT, tc=600.0)

# Configuración de 12 escenarios bidireccionales (+/-) y realización con ruido
VARS = {
    "Q_stock": ("Q_stock", "rel", 0.10),
    "T_feed": ("T alim.", "add", 5.0),
    "CP_stock": ("ClO4 alim.", "rel", 0.25),
    "C_stock": ("KNO3 alim.", "rel", 0.05),
    "T_dil": ("T agua dil.", "add", 10.0),
    "valve_gain": ("Válvula dil.", "rel", 0.20),
}
COLV = dict(zip(VARS, ["tab:blue", "tab:red", "tab:green", "tab:purple", "tab:orange", "tab:brown"]))
ESC, COL = {}, {"Solo ruido": "k"}
for k, (lab, kind, mag) in VARS.items():
    for sg in (+1, -1):
        nom = f"{lab} {sg*mag*100:+.0f} %" if kind == "rel" else f"{lab} {sg*mag:+.0f} °C"
        ESC[nom] = (k, kind, mag, sg)
        COL[nom] = COLV[k]
AUM = [n for n, v in ESC.items() if v[3] > 0]
DIS = [n for n, v in ESC.items() if v[3] < 0]

def hacer_dist(key, kind, mag, signo, ruido=None):
    def f(t):
        d = dict(d0)
        if t >= TP:
            d[key] = d.get(key, 1.0) * (1.0 + signo * mag) if kind == "rel" else d[key] + signo * mag
        return ruido(d, t) if ruido is not None else d
    return f

DIST = {n: hacer_dist(*v) for n, v in ESC.items()}
DIST_R = {n: hacer_dist(*v, ruido=RUIDO) for n, v in ESC.items()}
DIST_R["Solo ruido"] = lambda t: RUIDO(dict(d0), t)

def lazo_abierto(dist, H=H_OL, dt=DT, log=12):
    return simulate(pl, x0, H, dt, lambda t, x, pl_: u0, disturbance=dist, log_every=log)

def lazo_cerrado(dist, H=H_OL, dt=DT, log=12, tuning=tun, enabled=("T", "V", "ratio"), consigna=None):
    qsf = (lambda t: dist(t)["Q_stock"]) if dist is not None else (lambda t: qs0)
    ctrl = C.PlantController(pl, u0, sp, tuning, enabled=enabled, q_stock_meas=qsf, ratio_nom=R0)
    if consigna:
        ctrl = ConsignaVariable(ctrl, consigna)
    return simulate(pl, x0, H, dt, ctrl, disturbance=dist, log_every=log), ctrl

# %% [markdown]
# ## Parte 1 y 2 · Lazo Abierto vs Lazo Cerrado (Determinista y con Ruido Coloreado)

# %%
# 1.1 Verificación de reposo estacionario e identificación FOPDT (+/- 10% m_evap)
df0 = lazo_abierto(None, H=6 * 3600, log=60)
assert max(float(np.max(np.abs(df0[k] - df0[k].iloc[0]))) for k in ("V", "T", "C", "CP")) < 1e-6

OL = {n: lazo_abierto(f) for n, f in DIST.items()}
CL = {n: lazo_cerrado(f)[0] for n, f in DIST.items()}
OL_R = {n: lazo_abierto(f) for n, f in DIST_R.items()}
CL_R = {n: lazo_cerrado(f)[0] for n, f in DIST_R.items()}

# Consistencia modelo-controlador a 24 h (offset nulo por acción integral)
for n in ("T alim. +5 °C", "T alim. -5 °C", "Q_stock +10 %", "Q_stock -10 %"):
    f = DIST[n]
    dfc, _ = lazo_cerrado(f, H=24 * 3600, log=60)
    d_end = f(24 * 3600)
    xs, us, ds = pl.steady_state(alpha=0.25, d=d_end, Q_dil=R0 * d_end["Q_stock"])
    ms = pl.metrics(xs, us, ds)
    assert 100.0 * abs(dfc.recovery_pct.iloc[-1] / ms["recovery_pct"] - 1.0) < 0.1

# %% [markdown]
# ## Parte 3 · Seguimiento de Consignas, Barrido IMC (ε), Ponderación β y Frente de Pareto

# %%
H_SP = 8 * 3600
K_, tau_, th_ = info["T"]["K"], info["T"]["tau_s"], max(info["T"]["theta_s"], 60.0)
fT_step = lambda t: x0[1] - (2.0 if t >= TP else 0.0)

# Barrido de filtro IMC (ε) para construir el Frente de Pareto (Variabilidad T vs Esfuerzo m_evap)
pareto_rows = []
for eps_frac in (0.05, 0.10, 0.20, 0.35, 0.60, 1.00):
    tt = imc_pi(K_, tau_, th_, eps_frac * tau_)
    df_p, _ = lazo_cerrado(DIST_R["Solo ruido"], H=H_OL, tuning={**tun, "T": tt})
    pareto_rows.append({
        "eps_tau": eps_frac,
        "Kp": tt.Kp,
        "sigma_T_C": sigma_ventana(df_p, "T"),
        "sigma_m_evap_kg_s": float(df_p.u_m_evap.iloc[len(df_p)//2:].std()),
        "IAE_T_ruido": iae(df_p.t_h, df_p["T"] - x0[1]),
    })
df_pareto = pd.DataFrame(pareto_rows)
print(df_pareto.round(4))

# Análisis de Estabilidad Routh-Hurwitz y Bifurcación de Hopf del lazo de temperatura
rh_info = routh_hopf_analysis(Kp=K_, tau_p=tau_, theta=max(info["T"]["theta_s"], 120.0), tau_I=tau_)
print("Límites Routh-Hopf lazo T:", {k: round(v, 4) for k, v in rh_info.items()})

# %% [markdown]
# ## Parte 4 y Parte B · Acoplamiento NPT-3 (Áreas 2000–3000–6000) y Lazo AIT 3105 + AIC 3255

# %%
df_osc = simulate_plant_npt3_calibrated(use_AIT3105=True, use_AIC3255=True, tuning_mode="ZN_oscillatory")
df_tl = simulate_plant_npt3_calibrated(use_AIT3105=True, use_AIC3255=True, tuning_mode="TL_damped")
df_hopf = simulate_plant_npt3_calibrated(use_AIT3105=True, use_AIC3255=True, tuning_mode="Hopf_limit_cycle")

df_sin_AIT = simulate_plant_npt3_calibrated(use_AIT3105=False, use_AIC3255=False, tuning_mode="TL_damped")
df_con_AIT = simulate_plant_npt3_calibrated(use_AIT3105=True, use_AIC3255=False, tuning_mode="TL_damped")
df_con_ambos = simulate_plant_npt3_calibrated(use_AIT3105=True, use_AIC3255=True, tuning_mode="TL_damped")

print("Pico ClO4 producto final Área 6000 SIN AIT 3105:", round(df_sin_AIT["ppm_ClO4_final_6000"].max(), 2), "ppm")
print("Pico ClO4 producto final Área 6000 CON AIT 3105:", round(df_con_AIT["ppm_ClO4_final_6000"].max(), 2), "ppm")
print("Pico ClO4 producto final Área 6000 CON AIT 3105 + AIC 3255:", round(df_con_ambos["ppm_ClO4_final_6000"].max(), 2), "ppm")

## 6. Integración con el Cuaderno 06 y Flujo de Trabajo en el Repositorio

Para mantener la separación modular definida en **ROADMAP_control_por_etapas.md** y **control_PID_P&ID.md** dentro de **CristalizadorPT_add**:

* **En notebooks/05_Control_PID_dinamico.ipynb:** Se ejecuta el flujo completo de validación en lazo abierto ($\pm$ y con ruido), lazo cerrado regulatorio, seguimiento de consignas ($\varepsilon$, $\beta$), estabilidad de Routh-Hopf y el frente de Pareto entre $\sigma_T$ y $\sigma_{m_{\text{evap}}}$.


* **En notebooks/06_Control_por_etapas_dilucion.ipynb:** Se importa directamente `simulate_plant_npt3_calibrated` desde **msmpr_core.plant** junto con el bloque de la Parte B (secciones B1 a B4 de **05_control_pid_perturbaciones.ipynb**) para evaluar específicamente la sensibilidad al tiempo muerto $\theta_{\text{AIT}} \in [0, 120]\text{ min}$, la saturación de la válvula de agua ($3\times$ vs $8\times$) y la acción conjunta con la válvula de purga **FV 3522** gobernada por **AIC 3255**.
